# DSA Week 10 -- Graphs: BFS & DFS

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Prerequisites:** Weeks 1-9
**Focus:** Graph representation, BFS, DFS

## Learning Objectives

1. Represent a graph using adjacency lists
2. Implement BFS (Breadth-First Search) and DFS (Depth-First Search)
3. Trace both algorithms step-by-step
4. Understand when to use BFS vs DFS
5. Apply graphs to real-world problems in your track

## The Big Idea

A graph models connections between things. Nodes (vertices) are the things,
edges are the connections. Graphs are EVERYWHERE:

```
Examples:
  - Social network: people (nodes) connected by friendships (edges)
  - Road map: cities (nodes) connected by roads (edges)
  - Internet: computers (nodes) connected by cables (edges)
  - Pipeline: stages (nodes) connected by data flow (edges)
  - Sensor network: devices (nodes) connected by communication links

Graph:
    A --- B
    |     |
    C --- D --- E

Adjacency list representation:
    A: [B, C]
    B: [A, D]
    C: [A, D]
    D: [B, C, E]
    E: [D]
```

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Graph Representation

In [ ]:
class Graph:
    """Graph using adjacency list representation."""

    def __init__(self, directed=False):
        self.adj = {}
        self.directed = directed

    def add_node(self, node):
        if node not in self.adj:
            self.adj[node] = []

    def add_edge(self, u, v, weight=None):
        """Add edge between u and v."""
        self.add_node(u)
        self.add_node(v)
        self.adj[u].append((v, weight) if weight is not None else v)
        if not self.directed:
            self.adj[v].append((u, weight) if weight is not None else u)

    def neighbors(self, node):
        """Return neighbors of node."""
        return self.adj.get(node, [])

    def nodes(self):
        return list(self.adj.keys())

    def show(self):
        """Print adjacency list."""
        print("  Graph (" + ("directed" if self.directed else "undirected") + "):")
        for node in sorted(self.adj.keys(), key=str):
            neighbors = self.adj[node]
            print("    " + str(node) + " -> " + str(neighbors))

# Build a graph
g = Graph()
edges = [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E")]
for u, v in edges:
    g.add_edge(u, v)

g.show()

---
## Part 2: BFS (Breadth-First Search)

BFS explores the graph **level by level**, like ripples on water.
It finds the **shortest path** (fewest edges) from start to any node.

```
BFS from A:

  Level 0: A
  Level 1: B, C    (neighbors of A)
  Level 2: D       (neighbors of B, C not yet visited)
  Level 3: E       (neighbor of D not yet visited)

  Order: A -> B -> C -> D -> E
```

In [ ]:
from collections import deque

def bfs_traced(graph, start):
    """BFS with step-by-step trace. O(V + E)."""
    visited = set()
    queue = deque([start])
    visited.add(start)
    order = []
    level = {start: 0}

    print("  BFS from " + str(start) + ":")
    print()

    while queue:
        node = queue.popleft()
        order.append(node)
        lvl = level[node]
        neighbors = graph.neighbors(node)
        new_neighbors = [n for n in neighbors if n not in visited]

        print("    Visit " + str(node) + " (level " + str(lvl) + ") | queue=" + str(list(queue)) + " | new neighbors=" + str(new_neighbors))

        for neighbor in neighbors:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
                level[neighbor] = lvl + 1

    print()
    print("  BFS order: " + " -> ".join(str(n) for n in order))
    return order

bfs_traced(g, "A")

---
## Part 3: DFS (Depth-First Search)

DFS explores the graph by going as **deep as possible** before backtracking.
It uses a stack (or recursion).

```
DFS from A (exploring alphabetically):

  Visit A -> go to B (first neighbor)
    Visit B -> go to D (first unvisited neighbor)
      Visit D -> go to C (first unvisited neighbor)
        Visit C -> no unvisited neighbors, BACKTRACK
      Back at D -> go to E
        Visit E -> no unvisited neighbors, BACKTRACK

  Order: A -> B -> D -> C -> E
```

In [ ]:
def dfs_traced(graph, start):
    """DFS with step-by-step trace. O(V + E)."""
    visited = set()
    order = []

    def _dfs(node, depth):
        if node in visited:
            return
        visited.add(node)
        order.append(node)
        indent = "    " + "  " * depth
        neighbors = graph.neighbors(node)
        unvisited = [n for n in neighbors if n not in visited]
        print(indent + "Visit " + str(node) + " | unvisited neighbors: " + str(unvisited))

        for neighbor in neighbors:
            if neighbor not in visited:
                _dfs(neighbor, depth + 1)

        if not unvisited:
            print(indent + "(backtrack from " + str(node) + ")")

    print("  DFS from " + str(start) + ":")
    print()
    _dfs(start, 0)
    print()
    print("  DFS order: " + " -> ".join(str(n) for n in order))
    return order

dfs_traced(g, "A")

---
## Part 4: BFS vs DFS -- When to Use Which

| Feature | BFS | DFS |
|---------|-----|-----|
| Explores | Level by level | Deep first |
| Data structure | Queue (FIFO) | Stack (LIFO) / recursion |
| Shortest path (unweighted) | YES | NO |
| Memory | O(width of graph) | O(depth of graph) |
| Good for | Finding shortest path, level-order | Checking connectivity, topological sort |
| Finds all nodes? | YES | YES |
| Time complexity | O(V + E) | O(V + E) |

In [ ]:
# Side-by-side comparison on a larger graph
g2 = Graph()
for u, v in [("1", "2"), ("1", "3"), ("2", "4"), ("2", "5"),
             ("3", "6"), ("3", "7"), ("4", "8"), ("5", "8"),
             ("6", "9"), ("7", "9")]:
    g2.add_edge(u, v)

print("=== Graph ===")
g2.show()
print()
print("=== BFS ===")
bfs_order = bfs_traced(g2, "1")
print()
print("=== DFS ===")
dfs_order = dfs_traced(g2, "1")
print()
print("BFS visits level-by-level (shortest paths).")
print("DFS goes deep before backtracking.")

---
## Part 5: Practical Application -- Finding Connected Components

In [ ]:
def find_connected_components(graph):
    """Find all connected components using BFS. O(V + E)."""
    visited = set()
    components = []

    for node in graph.nodes():
        if node not in visited:
            # BFS to find all nodes in this component
            component = []
            queue = deque([node])
            visited.add(node)
            while queue:
                current = queue.popleft()
                component.append(current)
                for neighbor in graph.neighbors(current):
                    if neighbor not in visited:
                        visited.add(neighbor)
                        queue.append(neighbor)
            components.append(component)

    return components

# Graph with disconnected components
g3 = Graph()
g3.add_edge("A", "B")
g3.add_edge("B", "C")
g3.add_edge("D", "E")  # separate component
g3.add_node("F")        # isolated node

components = find_connected_components(g3)
print("Connected components:")
for i, comp in enumerate(components):
    print("  Component " + str(i + 1) + ": " + str(comp))

---
## Mini-Quiz

In [ ]:
# Q1: You need to find the shortest route between two cities.
# Use BFS or DFS? Why?
# Answer:

# Q2: You want to check if a network of sensors is fully connected.
# Use BFS or DFS? Does it matter?
# Answer:

# Q3: What is the time complexity of BFS and DFS?
# Answer:

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)